# ISOM 839 · Session 10 — Markov Decision Processes: Deciding Over Time

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last session's chains told us what *happens*. Tonight we take the wheel: **states, actions, rewards, policies**. The Bellman equation — the value of a state is the best you can do now plus the discounted value of where that choice takes you — and value iteration, the algorithm that solves it. The playground is the classic inventory problem, and your code is about to rediscover 1950s inventory theory on its own.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Part 1 — The inventory MDP (tonight's lab)

A small shop. **State:** units on hand this morning, 0–10 (the shelf holds 10). **Action:** order 0–5 units, delivered instantly. **Demand:** 0, 1, 2 or 3 units, equally likely. **Rewards:** $10 margin per unit sold, $8 per unit ordered, $15 goodwill penalty per unit of unmet demand. Discount γ = 0.95 per day.

**Step 1 — define it.** Everything above is five numbers and an array.

In [ ]:
MAX_INV, MAX_ORDER = 10, 5
demands = np.array([0, 1, 2, 3]); probs = np.full(4, 0.25)
PRICE, COST, PENALTY, GAMMA = 10, 8, 15, 0.95

**Step 2 — the Bellman update.** For each state s and action a: expected immediate reward, plus γ times the expected value of tomorrow's inventory, averaged over demand. Take the best action. That is the whole algorithm.

**Step 3 — iterate to convergence.** Sweep all states until the largest change is below 1e-6.

In [ ]:
def value_iteration(PRICE=PRICE, COST=COST, PENALTY=PENALTY, GAMMA=GAMMA, HOLD=0.0, tol=1e-6):
    V = np.zeros(MAX_INV + 1)
    for sweep in range(1, 100_000):
        V_new = np.full(MAX_INV + 1, -np.inf); policy = np.zeros(MAX_INV + 1, dtype=int)
        for s in range(MAX_INV + 1):
            for a in range(min(MAX_ORDER, MAX_INV - s) + 1):
                stock, q = s + a, 0.0
                for d, p in zip(demands, probs):
                    sold = min(stock, d); left = stock - sold
                    reward = PRICE * sold - COST * a - PENALTY * (d - sold) - HOLD * left
                    q += p * (reward + GAMMA * V[left])
                if q > V_new[s] + 1e-12:
                    V_new[s], policy[s] = q, a
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, policy, sweep
        V = V_new

V, policy, sweeps = value_iteration()
print(f'converged in {sweeps} sweeps')
print(' stock  order  order-up-to   value')
for s in range(MAX_INV + 1):
    print(f'   {s:2d}     {policy[s]}        {s + policy[s]:2d}      {V[s]:7.2f}')

**Step 4 — read the policy.** Below some stock level the shop orders *up to* a target; above it, nothing. Inventory theorists proved this **(s, S)** structure is optimal in the 1950s. Your value iteration just rediscovered it numerically — nobody told it to.

**Discounting changes convergence, not the policy.** Try it:

In [ ]:
for g in (0.80, 0.95, 0.99):
    V_g, pol_g, sw = value_iteration(GAMMA=g)
    print(f'gamma {g}: {sw:4d} sweeps · policy {list(map(int, pol_g))} · V[0] = {V_g[0]:.1f}')

### 💬 Discussion
Raise the stockout penalty to $40 and re-run — which way does the order-up-to level move? Now cut the margin to $9 (barely above the $8 cost). At what point does the shop stop ordering altogether? *The parameters are the policy.* A manager who can name which cost drives the threshold understands the business; one who cannot is guessing.

### Powell's bridge
Operations research, control theory and reinforcement learning all solve this same problem with different vocabulary. Warren Powell's four classes of policies — policy function approximations, cost function approximations, value function approximations (what you just did), direct lookahead — are the Rosetta stone. Next session: what happens when the transition probabilities are *unknown* and the agent has to learn them.

---
# Homework #7 · Option A — Inventory MDP with Holding Costs 📦

**Pick one of the two Homework #7 options on Canvas: this one, or the pricing curve in the Session 8 notebook. Same points, same rubric.** Due date on Canvas.

A different shop with a wider demand distribution, so that the holding cost has room to matter. **State:** 0–12 units. **Action:** order 0–6. **Demand:** 0–5 units with probabilities 0.10, 0.20, 0.30, 0.20, 0.15, 0.05. **Margin $12, order cost $7, stockout penalty $6, γ = 0.95.** The parameters are in the cell below; the `value_iteration` function above already accepts a `HOLD` argument — read it to see where the holding charge enters.

1. **Baseline vs holding cost.** Run value iteration with no holding cost, then with **$1 per unit of leftover inventory per day** (the real cost of tied-up capital and shelf space). Print the two policies side by side.
2. **How did the order-up-to threshold move, and why?** Two or three sentences.
3. **Sweep the holding cost from $0 to $4** in steps of $0.50 and plot the order-up-to level (the largest stock-after-ordering the policy ever chooses) against it.
4. **One paragraph for a store manager:** in plain English, what trade-off is the model balancing, and when would carrying "too much" stock actually be the right call?

In [ ]:
MAX_INV, MAX_ORDER = 12, 6
demands = np.arange(6); probs = np.array([0.10, 0.20, 0.30, 0.20, 0.15, 0.05])
PRICE, COST, PENALTY, GAMMA = 12, 7, 6, 0.95

# TODO 1: value_iteration(...) with HOLD=0 and HOLD=1; print both policies and order-up-to levels side by side

# TODO 2: answer in a markdown cell — which way the threshold moved and why

# TODO 3: sweep HOLD over np.arange(0, 4.01, 0.5); plot order-up-to level vs holding cost

# TODO 4: the manager paragraph in a markdown cell below (and in the Canvas text box, starting with "Option A")

---
### Submission checklist (Option A)
- [ ] Both policies printed side by side, order-up-to levels stated
- [ ] The threshold explanation (which cost it trades against)
- [ ] The sweep plot
- [ ] The manager paragraph — pasted into the Canvas text box, starting with "Option A"

**Next class — Session 11 on December 2** (no class November 25, Thanksgiving): deep learning meets optimization — predict, then decide. 🧠